# FlyLM-3-overfit-test-training

**Question:** can an image of **A, B, or C** travel through the full MaleCNS circuit and
make the right-foreleg motor readout draw that same letter?

`A/B/C image → mapped R1–R6 → full recurrent MaleCNS → right-foreleg motor activity → pen trace`

This is a **same-character copy / overfit test**, not next-character language modeling.
Random independent next letters would be unpredictable; repeating `ABCABC` with carried
state would allow a sequence shortcut. Here every example starts with **zero neural and
retinal state**, and every GPU gets one of each letter in shuffled order per update.
Only images enter the model. Labels select supervision and never enter the circuit.

The notebook preserves the current FlyLM-3 full `Traced` graph, retinal mapping, mixed
spiking/graded dynamics, right-foreleg selection, two-link kinematics and motor-only
linear pen head. It starts a **new experiment**; the original notebooks are unchanged.
The leg remains an engineered kinematic surrogate with no muscle/contact physics.

Run top to bottom with a GPU and Internet (Kaggle supported). The initial graph download
is approximately 1.1 GB; the reusable graph cache and checkpoints need several GB of disk.
All implementation and diagnostics are inside this notebook.

**Success requires more than classification:** all three actual pen traces must fit,
including pen-up transitions, and blank/constant images or clamped motor signals must
lose discrimination. Passing on three memorized glyphs does not demonstrate language
learning or robustness to new fonts. Both the readout-only and end-to-end results are saved.

## 1. Configuration and reproducibility

The first stage fits the pen head to **cached motor activity computed through the full
untrained circuit**, without changing the circuit. This isolates whether its visual signal
is usable. The second stage recomputes the full circuit and trains all parameters. Separate
learning rates and gradient norms make circuit instability visible.

The conservative end-to-end rates (`lr=1e-7` for the circuit, `joint_head_lr=3e-4`)
were chosen after a local ABC probe: rates from `1e-4` to `1e-6` disrupted the fitted
spiking representation, whereas `1e-7` preserved the copy fit for 30 updates. This
is calibration on the overfit set, not a physiological setting or a universal optimizer recipe.

The original pen MSE can reward an average of uncertain next letters. Copy supervision
removes that ambiguity. Position error is normalized by the mean-trace baseline, pen-down
and pen-up examples receive equal aggregate contact weight, and template cross entropy
is **reported only**: its distance-based gradient need not vanish at an exact target trace.

Defaults keep **16 ticks per independent image**, rather than assuming more ticks help.
The latency diagnostic measures A/B/C separation at 4, 8, 16, 32 and 64 ticks. Increase
`brain_steps` only if the reported propagation evidence warrants it. Unknown activity
remains spiking. `graded_control` is an explicit experiment choice, not an automatic fallback.

The inherited P100 bootstrap checks CUDA support before importing torch in this kernel.
Only configuration here controls the run. Early stopping requires the complete fit and
intervention criteria for multiple evaluations. The optional ZIP export is disabled by default.

**Kaggle: select GPU T4 ×2.** The full training stage uses both GPUs through PyTorch
data parallelism: three examples per GPU, global batch six. Model/graph parameters are
replicated, predictions are gathered, and one global mean loss averages both replicas'
gradients. The small cached-head stage and evaluations run on GPU 0. If only one GPU
is available the notebook reports the fallback and uses batch three. This is data
parallelism, so each T4 must hold a complete graph; the two VRAM capacities are not pooled.
With only three unique images, this is a correctness test, not a throughput benchmark.

In [ ]:
import importlib.util, subprocess, sys, json
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Select a runtime with PyTorch installed (Kaggle GPU recommended).')

def disk_cuda_info():
    # Import torch only in a child process until the installed build is ready.
    probe = subprocess.run([sys.executable, '-c',
        "import torch,json; print(json.dumps({'version':str(torch.__version__),'cap':list(torch.cuda.get_device_capability()) if torch.cuda.is_available() else None,'arch':torch.cuda.get_arch_list()}))"],
        capture_output=True, text=True, check=True)
    return json.loads(probe.stdout.strip().splitlines()[-1])

disk_info = disk_cuda_info()
print('Installed PyTorch:', disk_info)
if disk_info['cap'] == [6, 0] and 'sm_60' not in disk_info['arch']:
    print('Installing the CUDA 12.6 build with P100 support before importing torch.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade',
        'torch==2.10.0+cu126', 'torchvision==0.25.0+cu126', 'torchaudio==2.10.0+cu126',
        '--index-url', 'https://download.pytorch.org/whl/cu126'])
    importlib.invalidate_caches()
    disk_info = disk_cuda_info()
    print('After installation:', disk_info)
    if disk_info['cap'] != [6, 0] or 'sm_60' not in disk_info['arch']:
        raise RuntimeError('P100 support was not verified after installation. Check the pip output and GPU settings.')

# Like flyLM-2: a fresh kernel can continue immediately after installation.
# Do not remove/reload torch modules: native libraries already loaded cannot be safely swapped.
loaded_torch = sys.modules.get('torch')
if (disk_info['cap'] == [6, 0] and loaded_torch is not None
        and 'sm_60' not in loaded_torch.cuda.get_arch_list()):
    raise RuntimeError(
        'An incompatible torch build was already imported before this setup cell. '
        'Its live CUDA libraries cannot be safely replaced in place. '
        'Run this setup before any torch imports in a fresh kernel; no session restart '
        'is required when this setup runs first.'
    )
required = {'numpy':'numpy', 'pandas':'pandas', 'scipy':'scipy', 'pyarrow':'pyarrow',
            'tqdm':'tqdm', 'PIL':'pillow', 'matplotlib':'matplotlib'}
missing = [v for k,v in required.items() if importlib.util.find_spec(k) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import gc, io, hashlib, math, platform, random, shutil, time, urllib.request
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import scipy
from scipy import sparse
from scipy.sparse.csgraph import dijkstra
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties, findfont
from PIL import Image, ImageDraw

CFG = dict(seed=42, n_neurons=None, brain_steps=16, batch_size=6, tbptt=1,
    train_updates=200, lr=0.0000001, neuron_mode='mixed', unknown_activity='spiking',
    spike_threshold_center=0.01, spike_threshold_radius=0.008, spike_reset=0.0,
    refractory_ticks=2, surrogate_slope=5.0, checkpoint_tokens=True,
    nt_confidence=0.8, use_nt_signs=True, edge_block=262144,
    tau_initial=(2.0,2.0,64.0), tau_min=(0.5,0.5,8.0), tau_max=(32.0,16.0,512.0),
    log_every=10, eval_every=10, save_every=50, eval_chunks=1,
    vram_fraction=0.90, image_size=48, pen_steps=128,
    retina_tau_ticks=4.0, retinal_blur=3, retinal_gain=1.0, retinal_half_saturation=0.02,
    # Intrinsic resting offset for graded lamina cells; no image information enters here.
    lamina_rest=0.30, link_lengths=(0.8,0.8),
    joint_low=(-1.8,0.2), joint_high=(1.0,2.8),
    ink_loss_weight=0.2, language_loss_weight=0.0, template_temperature=0.03,
    head_updates=300, head_lr=0.01, joint_head_lr=0.0003, requested_gpus=2,
    examples_per_class=64, eval_examples_per_class=16,
    fit_relative_mse=0.05, fit_max_class_relative_mse=0.10,
    fit_ink_recall=0.98, fit_penup_recall=0.95,
    early_stop_evals=3, export_zip=False)
WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()/'flylm3_work'
CACHE = WORK/'flylm_malecns_cache'
RUN_DIR = WORK/'flylm3_abc_overfit_runs'/datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
CACHE.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=False)
GPU_IDS = list(range(min(CFG['requested_gpus'], torch.cuda.device_count())))
device = torch.device('cuda:0' if GPU_IDS else 'cpu')
CFG['batch_size'] = 3 * max(1, len(GPU_IDS))
CFG['active_gpus'] = len(GPU_IDS)
if len(GPU_IDS) < CFG['requested_gpus']:
    print(f"Requested {CFG['requested_gpus']} GPUs; using {len(GPU_IDS)} with batch {CFG['batch_size']}.")
print('Training GPUs:', [dict(index=i, name=torch.cuda.get_device_name(i),
    memory_GiB=torch.cuda.get_device_properties(i).total_memory/2**30) for i in GPU_IDS])
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_all(CFG['seed'])
VERSIONS = dict(python=platform.python_version(), torch=str(torch.__version__),
    numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__,
    pyarrow=pa.__version__, matplotlib=matplotlib.__version__)
assert all(CFG[k] > 0 for k in ['batch_size','tbptt','brain_steps','train_updates','pen_steps',
    'log_every','eval_every','save_every','eval_chunks','retina_tau_ticks','template_temperature'])
assert 0 < CFG['vram_fraction'] < 1
assert isinstance(CFG['retinal_blur'],int) and CFG['retinal_blur']>0 and CFG['retinal_blur']%2==1
assert CFG['image_size']>1
r = torch.tensor([0,1,2], device=device)
c = torch.tensor([1,0], device=device)
w = torch.tensor([2.,3.], device=device)
torch.testing.assert_close(torch.sparse.mm(torch.sparse_csr_tensor(r,c,w,size=(2,2)),
    torch.ones(2,1,device=device)), torch.tensor([[2.],[3.]],device=device))
print(VERSIONS, 'device:', device, 'output:', RUN_DIR)
assert CFG['batch_size'] == 3*max(1, len(GPU_IDS)) and CFG['tbptt'] == 1
assert CFG['examples_per_class'] % max(1, len(GPU_IDS)) == 0
assert CFG['eval_examples_per_class'] % max(1, len(GPU_IDS)) == 0
assert CFG['language_loss_weight'] == 0, 'Template NLL is a diagnostic, not the copy objective.'
assert all(type(CFG[k]) is int and CFG[k] > 0 for k in
           ['head_updates', 'examples_per_class', 'eval_examples_per_class', 'early_stop_evals'])
assert CFG['head_lr'] > 0 and CFG['joint_head_lr'] > 0 and CFG['lr'] > 0
assert 0 < CFG['fit_relative_mse'] < 1 and 0 < CFG['fit_max_class_relative_mse'] < 1
assert 0 < CFG['fit_ink_recall'] <= 1 and 0 < CFG['fit_penup_recall'] <= 1
print('Resolved configuration:', json.dumps(CFG, indent=2))

## 2. A/B/C-only data and reachable pen targets

`abc_train.txt` and `abc_eval.txt` contain **only A, B and C**, with equal counts and
independently shuffled balanced triples. There is no Shakespeare download. The labels
are the displayed characters themselves; no one-character shift is applied.

There are exactly **three unique canonical images**, not hundreds of independent images.
Evaluation uses those three once each in a different order and reports per-letter results.
Repeated records in the saved dataset do not increase the claimed evaluation sample size.
The font-outline tracing, pen lifting, rasterization and inverse-kinematics checks are
inherited from FlyLM-3. A/B/C share one scale and display coordinate frame.

In [ ]:
chars = list('ABC')
char_to_index = {ch: i for i, ch in enumerate(chars)}

def balanced_records(repeats, seed):
    rng = np.random.default_rng(seed)
    replicas = max(1, len(GPU_IDS))
    return np.stack([np.concatenate([rng.permutation(3) for _ in range(replicas)])
                     for _ in range(repeats//replicas)]).astype(np.int64)

train_batches = balanced_records(CFG['examples_per_class'], CFG['seed'])
eval_batches = balanced_records(CFG['eval_examples_per_class'], CFG['seed'] + 1)
train_ids, val_ids = train_batches.ravel(), eval_batches.ravel()
train_text = ''.join(chars[i] for i in train_ids)
val_text = ''.join(chars[i] for i in val_ids)
assert set(train_text) == set(val_text) == set('ABC')
assert np.all(np.bincount(train_ids, minlength=3) == CFG['examples_per_class'])
assert np.all(np.bincount(val_ids, minlength=3) == CFG['eval_examples_per_class'])
(RUN_DIR/'abc_train.txt').write_text(train_text, encoding='utf-8')
(RUN_DIR/'abc_eval.txt').write_text(val_text, encoding='utf-8')
DATA_SPEC = dict(task='independent_image_to_same_character_trace', alphabet=chars,
    unique_train_images=3, unique_eval_images=3, train_records=len(train_text),
    eval_records=len(val_text), reset_state_every_example=True,
    train_sha256=hashlib.sha256(train_text.encode()).hexdigest(),
    eval_sha256=hashlib.sha256(val_text.encode()).hexdigest(),
    evaluation='same three canonical glyphs; order control; not held-out generalization')
font_path = Path(findfont(FontProperties(family='DejaVu Sans Mono'), fallback_to_default=False))
FONT_SPEC = dict(name=font_path.name, sha256=hashlib.sha256(font_path.read_bytes()).hexdigest(),
    whitespace={'space':'␣','newline':'↵'}, target='outline_contours_arc_length')
font = FontProperties(fname=str(font_path))

def glyph_contours(ch):
    visible = {' ':'␣', '\n':'↵'}.get(ch,ch)
    polygons = TextPath((0,0), visible, size=1, prop=font).to_polygons()
    result = []
    for p in polygons:
        if len(p) < 3: continue
        if not np.allclose(p[0],p[-1]): p = np.vstack([p,p[0]])
        if np.linalg.norm(np.diff(p,axis=0),axis=1).sum() > 1e-8: result.append(p)
    if not result: raise ValueError(f'No drawable contours: {ch!r}')
    return result
all_contours = [glyph_contours(ch) for ch in chars]
all_xy = np.concatenate([p for contours in all_contours for p in contours])
lo, hi = all_xy.min(0), all_xy.max(0)
scale = 0.8 / max(hi-lo)
center = (lo+hi)/2

def make_trace(contours, steps):
    lengths = np.array([np.linalg.norm(np.diff(p,axis=0),axis=1).sum() for p in contours])
    # Each contour gets >=3 ink segments plus one pen-up start.
    assert steps >= 4*len(contours)
    extra = steps-4*len(contours)
    alloc = np.floor(extra*lengths/lengths.sum()).astype(int)+4
    for i in np.argsort(-(extra*lengths/lengths.sum()%1))[:steps-alloc.sum()]: alloc[i] += 1
    trace = []
    for poly, count in zip(contours, alloc):
        dist = np.r_[0,np.cumsum(np.linalg.norm(np.diff(poly,axis=0),axis=1))]
        keep = np.r_[True,np.diff(dist)>1e-10]
        poly,dist = poly[keep],dist[keep]
        sample = np.linspace(0,dist[-1],count)
        xy = np.stack([np.interp(sample,dist,poly[:,j]) for j in range(2)],axis=-1)
        xy = (xy-center)*scale+np.array([0.8,0.0])
        ink = np.ones(count); ink[0]=0
        trace.append(np.column_stack([xy,ink]))
    return np.concatenate(trace).astype(np.float32)

def forward_kinematics(q, lengths):
    q1,q2 = q.unbind(-1); l1,l2 = lengths
    return torch.stack([l1*q1.cos()+l2*(q1+q2).cos(),
                        l1*q1.sin()+l2*(q1+q2).sin()],dim=-1)

def inverse_kinematics(xy, lengths):
    x,y = xy.unbind(-1); l1,l2 = lengths
    cos2 = (x*x+y*y-l1*l1-l2*l2)/(2*l1*l2)
    assert torch.all(cos2.abs() < 1), 'Glyph outside reachable workspace'
    q2 = cos2.acos()
    q1 = torch.atan2(y,x)-torch.atan2(l2*q2.sin(),l1+l2*q2.cos())
    return torch.stack([q1,q2],dim=-1)

def rasterize_trace(trace, size=None):
    size = CFG['image_size'] if size is None else size
    a = np.asarray(trace); oversample=3
    canvas = Image.new('L',(size*oversample,size*oversample),0)
    draw = ImageDraw.Draw(canvas)
    # Fixed display frame, x in [0.25,1.35], y in [-0.55,0.55].
    xy = np.column_stack([(a[:,0]-.25)/1.1, (.55-a[:,1])/1.1])*(size*oversample-1)
    for j in range(1,len(a)):
        if a[j,2] >= 0.5:
            draw.line([tuple(xy[j-1]),tuple(xy[j])],fill=255,width=oversample)
    return np.asarray(canvas.resize((size,size),Image.Resampling.LANCZOS),dtype=np.float32)/255

# Re-normalizing only ABC can enlarge glyphs past the inherited joint-angle limits.
# Keep one common scale, but shrink it until every trace has a small joint-limit margin.
requested_scale = scale
joint_margin = 0.02
for _ in range(40):
    target_trace_cpu = torch.from_numpy(np.stack([make_trace(p, CFG['pen_steps']) for p in all_contours]))
    target_joints_cpu = inverse_kinematics(target_trace_cpu[..., :2], CFG['link_lengths'])
    if (torch.all(target_joints_cpu > torch.tensor(CFG['joint_low']) + joint_margin)
            and torch.all(target_joints_cpu < torch.tensor(CFG['joint_high']) - joint_margin)):
        break
    scale *= 0.95
else:
    raise ValueError('Could not fit ABC targets within the configured joint limits.')
FONT_SPEC['normalization'] = dict(requested_scale=float(requested_scale), actual_scale=float(scale),
    center=center.tolist(), pen_origin=[0.8, 0.0], joint_margin=joint_margin)
print('Reachable shared glyph scale:', FONT_SPEC['normalization'])
torch.testing.assert_close(forward_kinematics(target_joints_cpu,CFG['link_lengths']),
    target_trace_cpu[...,:2],atol=2e-6,rtol=2e-6)
assert torch.all(target_joints_cpu > torch.tensor(CFG['joint_low']))
assert torch.all(target_joints_cpu < torch.tensor(CFG['joint_high']))
glyph_images_cpu = torch.from_numpy(np.stack([rasterize_trace(t) for t in target_trace_cpu.numpy()]))[:,None]
assert len(set(hashlib.sha256(x.numpy().tobytes()).hexdigest() for x in glyph_images_cpu)) == len(chars)
target_trace = target_trace_cpu.to(device); target_joints = target_joints_cpu.to(device)
glyph_images = glyph_images_cpu.to(device)

mean_xy = target_trace[..., :2].mean(0)
POSITION_SCALE = float((target_trace[..., :2] - mean_xy).square().mean())
assert POSITION_SCALE > 1e-6
CLASS_POSITION_SCALE = (target_trace[..., :2] - mean_xy).square().mean((1, 2)).clamp_min(1e-8)

def copy_batch(update):
    order = torch.tensor(train_batches[(update-1) % len(train_batches)], device=device)
    # B,T,C,H,W; each replica's consecutive triple contains one A, one B, one C.
    return glyph_images[order, None], order[:, None]

print(DATA_SPEC)
print('Mean-trace MSE normalization:', POSITION_SCALE)
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for i, ax in enumerate(axes):
    ax.imshow(glyph_images_cpu[i, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(chars[i]); ax.axis('off')
plt.tight_layout(); fig.savefig(RUN_DIR/'abc_targets.png', dpi=140); plt.show(); plt.close(fig)

## 3. Full anatomical graph and sensory/motor ports

These cells retain the current FlyLM-3 extraction and exact anatomical selectors. No nodes
or existing connections are pruned, and no new image-to-motor shortcut is introduced.

In [ ]:
BASE_URL = 'https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/'
SOURCES = {
    'annotations': ('body-annotations-male-cns-v1.0-minconf-0.5.feather', 14483314),
    'nt': ('body-neurotransmitters-male-cns-v1.0.feather', 43282834),
    'edges': ('connectome-weights-male-cns-v1.0-minconf-0.5.feather', 1051241946),
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()

def download_file(name, expected_size):
    target = CACHE / name
    if not target.exists() or target.stat().st_size != expected_size:
        if shutil.disk_usage(CACHE).free < expected_size + 512 * 1024**2:
            raise RuntimeError('데이터 다운로드에 필요한 디스크 공간이 부족합니다.')
        partial = target.with_suffix(target.suffix + '.part')
        with urllib.request.urlopen(BASE_URL + name, timeout=60) as response:
            with partial.open('wb') as f, tqdm(total=expected_size, unit='B', unit_scale=True, desc=name) as bar:
                while True:
                    chunk = response.read(8 * 1024**2)
                    if not chunk:
                        break
                    f.write(chunk)
                    bar.update(len(chunk))
        if partial.stat().st_size != expected_size:
            raise RuntimeError(f'파일 크기 불일치: {name}. v1.0 파일/다운로드를 확인하세요.')
        partial.replace(target)
    return target

files = {key: download_file(*spec) for key, spec in SOURCES.items()}
source_manifest = {key: dict(url=BASE_URL + p.name, bytes=p.stat().st_size, sha256=sha256_file(p))
                   for key, p in files.items()}
print({key: entry['bytes'] for key, entry in source_manifest.items()})

In [ ]:
def require_columns(names, required, label):
    absent = set(required)-set(names)
    if absent:
        raise ValueError(f'{label}: missing {sorted(absent)}; found {list(names)}')

def map_ids(sorted_ids, values):
    positions = np.searchsorted(sorted_ids, values)
    safe = positions.clip(0, len(sorted_ids)-1)
    return positions, (positions < len(sorted_ids)) & (sorted_ids[safe] == values)

annotation = feather.read_table(files['annotations']).to_pandas()
require_columns(annotation.columns, ['bodyId','status','superclass'], 'annotations')
neurons = annotation.loc[annotation.status == 'Traced'].sort_values('bodyId').set_index('bodyId')
assert neurons.index.is_unique and not neurons.index.hasnans
body_ids = neurons.index.to_numpy(np.int64)
n = len(body_ids)
CFG['n_neurons'] = n
assert n > 100_000, f'전체 Traced 데이터인지 확인하세요: {n}'
print('Actual full Traced neuron count:', n)
print(neurons['superclass'].value_counts(dropna=False).to_string())

def full_edge_batches():
    with pa.memory_map(str(files['edges']), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body_pre','body_post','weight'], 'edges')
        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)
            s, sm = map_ids(body_ids, batch.column('body_pre').to_numpy(zero_copy_only=False))
            d, dm = map_ids(body_ids, batch.column('body_post').to_numpy(zero_copy_only=False))
            keep = sm & dm
            if keep.any():
                w = batch.column('weight').to_numpy(zero_copy_only=False)[keep]
                if not np.isfinite(w).all() or (w <= 0).any():
                    raise ValueError('Invalid count')
                yield s[keep].astype(np.int32), d[keep].astype(np.int32), w.astype(np.int64)

extract_spec = dict(version=3, selection='all_status_Traced_no_component_filter',
                    orientation='post_row_pre_column', n=n,
                    source_sha256={k:v['sha256'] for k,v in source_manifest.items()})
cache_key = hashlib.sha256(json.dumps(extract_spec, sort_keys=True).encode()).hexdigest()[:16]
full_cache = CACHE / f'full_{cache_key}'
full_cache.mkdir(exist_ok=True)
manifest_path = full_cache / 'complete.json'
if not manifest_path.exists():
    degree = np.zeros(n, np.int64)
    for s, d, w in tqdm(full_edge_batches(), desc='Pass 1: full degrees'):
        degree += np.bincount(d, minlength=n)
    raw_edges = int(degree.sum())
    if raw_edges == 0:
        raise ValueError('No full-neuron edges')
    required_disk = raw_edges*24 + 512*1024**2
    if shutil.disk_usage(full_cache).free < required_disk:
        raise RuntimeError(f'CSR 구축 디스크가 부족합니다. 약 {required_disk/2**30:.1f} GiB 여유가 필요합니다.')
    crow = np.concatenate(([0], np.cumsum(degree)))
    col = np.lib.format.open_memmap(full_cache/'src.npy', mode='w+', dtype=np.int32, shape=(raw_edges,))
    count = np.lib.format.open_memmap(full_cache/'count.npy', mode='w+', dtype=np.int64, shape=(raw_edges,))
    cursor = crow[:-1].copy()
    for s, d, w in tqdm(full_edge_batches(), desc='Pass 2: write full CSR'):
        order = np.argsort(d, kind='stable')
        sd = d[order]
        starts = np.r_[0, np.flatnonzero(sd[1:] != sd[:-1])+1]
        sizes = np.diff(np.r_[starts, len(sd)])
        local = np.arange(len(sd))-np.repeat(starts, sizes)
        positions = cursor[sd]+local
        col[positions], count[positions] = s[order], w[order]
        cursor += np.bincount(d, minlength=n)
    assert np.array_equal(cursor, crow[1:])
    col.flush()
    count.flush()
    full = sparse.csr_matrix((count, col, crow), shape=(n,n), copy=False)
    full.sum_duplicates()
    full.sort_indices()
    nnz = int(full.nnz)
    # 중복 병합 후 유효한 배열 구간과 행 포인터를 명시적으로 저장한다.
    col[:nnz], count[:nnz] = full.indices, full.data
    col.flush()
    count.flush()
    np.save(full_cache/'crow.npy', full.indptr.astype(np.int64))
    np.save(full_cache/'body_ids.npy', body_ids)
    manifest_path.write_text(json.dumps(dict(spec=extract_spec, raw_edges=raw_edges, nnz=nnz)), encoding='utf-8')
    del full, col, count, cursor, degree, crow
    gc.collect()
full_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert full_manifest['spec'] == extract_spec
crow = np.load(full_cache/'crow.npy', mmap_mode='r')
src = np.load(full_cache/'src.npy', mmap_mode='r')[:full_manifest['nnz']]
counts = np.load(full_cache/'count.npy', mmap_mode='r')[:full_manifest['nnz']]
assert int(crow[-1]) == len(src) == len(counts)
assert np.array_equal(body_ids, np.load(full_cache/'body_ids.npy'))
graph_data = dict(body_ids=body_ids, crow=crow, src=src, counts=counts)
print(dict(neurons=n, edges=len(src), retained='all Traced-to-Traced pairs'))

In [ ]:
KNOWN_NT = {'acetylcholine', 'gaba', 'glutamate', 'histamine', 'dopamine', 'serotonin', 'octopamine'}

def attach_nt(body_ids):
    selected_parts = []
    wanted = np.sort(body_ids)
    with pa.memory_map(str(files['nt']), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body', 'ground_truth', 'predicted_nt',
                                              'predicted_nt_confidence', 'consensus_nt'], 'neurotransmitters')
        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)
            _, keep = map_ids(wanted, batch.column('body').to_numpy(zero_copy_only=False))
            if keep.any():
                selected_parts.append(batch.filter(pa.array(keep)).to_pandas())
    if not selected_parts:
        raise RuntimeError('선택된 body와 전달물질 테이블의 join이 비어 있습니다.')
    table = pd.concat(selected_parts, ignore_index=True)
    if table['body'].duplicated().any():
        raise ValueError('Duplicate NT body IDs: explicit aggregation is required')
    table = table.set_index('body').reindex(body_ids)
    truth = table['ground_truth'].fillna('').str.strip().str.lower()
    predicted = table['predicted_nt'].fillna('').str.strip().str.lower()
    confidence = pd.to_numeric(table['predicted_nt_confidence'], errors='coerce')
    trusted = truth.isin(KNOWN_NT)
    confident = predicted.isin(KNOWN_NT) & confidence.ge(CFG['nt_confidence']) & confidence.le(1)
    effective = np.where(trusted, truth, np.where(confident, predicted, 'unknown'))
    origin = np.where(trusted, 'ground_truth', np.where(confident, 'predicted_nt', 'unknown'))
    table['effective_nt'] = effective
    table['effective_source'] = origin
    table.index.name = 'body'
    return table



require_columns(neurons.columns, ['type','rootSide','assignedOlHex1','assignedOlHex2',
    'subclass','somaSide','somaNeuromere'], 'visual/motor annotations')
receptor_mask = neurons['type'].eq('R1-R6').to_numpy()
anchor_mask = (neurons['type'].isin(['L1','L2','L3']) &
    neurons['assignedOlHex1'].notna() & neurons['assignedOlHex2'].notna()).to_numpy()
votes = {}
for post in np.flatnonzero(anchor_mask):
    left,right = int(crow[post]),int(crow[post+1])
    ss,ww = src[left:right],counts[left:right]
    h = (float(neurons.iloc[post].assignedOlHex1),float(neurons.iloc[post].assignedOlHex2))
    for pre,w in zip(ss[receptor_mask[ss]],ww[receptor_mask[ss]]):
        d = votes.setdefault(int(pre),{}); d[h] = d.get(h,0)+int(w)
input_nodes = np.array(sorted(votes),dtype=np.int64)
assert len(input_nodes)>0, 'No mapped R1-R6 receptors'
hexes = np.array([max(votes[i], key=lambda h:(votes[i][h],h)) for i in input_nodes])
confidence = np.array([max(votes[i].values())/sum(votes[i].values()) for i in input_nodes])
xy = np.column_stack([hexes[:,0]-.5*hexes[:,1],np.sqrt(3)/2*hexes[:,1]])
sides = neurons.iloc[input_nodes].rootSide.to_numpy()
assert set(sides) == {'L','R'}, 'Retinal root sides must be explicitly L/R'
uv = np.zeros_like(xy,dtype=np.float32)
for side in ['L','R']:
    mask = sides == side; points=xy[mask]; span=np.ptp(points,axis=0)
    assert (span>0).all()
    z=(points-points.min(0))/span
    uv[mask,0] = .6*z[:,0] if side=='L' else .4+.6*(1-z[:,0])
    uv[mask,1] = 1-z[:,1]
motor_mask = (neurons.superclass.eq('vnc_motor') & neurons.subclass.eq('fl') &
    neurons.somaSide.eq('R') & neurons.somaNeuromere.eq('T1')).to_numpy()
output_nodes = np.flatnonzero(motor_mask).astype(np.int64)
lamina_nodes = np.flatnonzero(neurons['type'].isin(['L1','L2','L3','L5'])).astype(np.int64)
assert len(output_nodes)>0, 'No right foreleg motor annotations; no fallback is permitted'
assert neurons.iloc[input_nodes].superclass.eq('ol_sensory').all()
assert not np.intersect1d(input_nodes,output_nodes).size
nt_table = attach_nt(body_ids)
# Histamine is explicitly inhibitory in this visual proxy; other uncertain signs retain v2 freedom.
node_sign = nt_table.effective_nt.map({'acetylcholine':1.,'gaba':-1.,'histamine':-1.}).fillna(0).to_numpy(np.float32)
if not CFG['use_nt_signs']: node_sign[:] = 0
edge_sign = node_sign[src]
adj_post_pre = sparse.csr_matrix((np.ones(len(src),np.uint8),src,crow),shape=(n,n))
from_retina = dijkstra(adj_post_pre.T.tocsr(),directed=True,indices=input_nodes,min_only=True,unweighted=True)
motor_dist = from_retina[output_nodes]
assert np.isfinite(motor_dist).any(), 'No directed retinal → right-foreleg path'
GRAPH_SPEC = dict(neurons=n, edges=len(src), mapped_receptors=len(input_nodes),
    unmapped_receptors=int(receptor_mask.sum()-len(input_nodes)), right_foreleg_motors=len(output_nodes),
    reachable_motors=int(np.isfinite(motor_dist).sum()), shortest_motor_path=float(np.min(motor_dist)),
    median_retina_confidence=float(np.median(confidence)),
    foreleg_selector='vnc_motor & fl & R somaSide & T1 somaNeuromere')
neurons.iloc[output_nodes].to_csv(RUN_DIR/'right_foreleg_motors.csv')
pd.DataFrame(dict(bodyId=body_ids[input_nodes],u=uv[:,0],v=uv[:,1],confidence=confidence)).to_csv(
    RUN_DIR/'retinal_projection.csv',index=False)
print(GRAPH_SPEC)
print(neurons.iloc[output_nodes][['type','subclass','somaSide','somaNeuromere']].to_string())
del adj_post_pre, votes, annotation
gc.collect()

def dynamics_types(frame):
    labels = pd.Series('', index=frame.index, dtype=object)
    for column in ['type', 'class', 'superclass']:
        if column not in frame:
            continue
        values = frame[column].fillna('').astype(str).str.strip()
        valid = ~values.str.lower().isin(['', 'none', 'nan', 'unknown', 'tbc', 'untyped'])
        use = labels.eq('') & valid
        labels.loc[use] = column + ':' + values.loc[use]
    labels = labels.mask(labels.eq(''), 'unknown')
    codes, names = pd.factorize(labels, sort=True)
    return codes.astype(np.int64), list(names)

type_ids, type_names = dynamics_types(neurons)
print('Dynamics groups:', len(type_names), 'type-annotated neurons:',
      sum(type_names[i].startswith('type:') for i in type_ids))

## 4. Inherited mixed dynamics and evidence registry

The original exact-type assignments, uncertainty annotations, NT-sign handling and
dimensionless excitability choices are preserved. Their original sources remain attached.

In [ ]:
# 문헌의 세포 유형을 MaleCNS의 정확한 type 문자열에 연결하는 작은 근거집이다.
EVIDENCE_VERSION = 'flylm3_activity_evidence_v1'


ACTIVITY_EVIDENCE = [
    dict(type_name='H2', activity='spiking',
         evidence_level='recorded_type_transfer', species='Drosophila melanogaster',
         preparation='성체; H2 전세포 기록', sex='이 근거집에서는 성별을 확정하지 않음',
         source='https://doi.org/10.1038/s41586-025-09276-5',
         locator='H2 electrophysiology; 국소 운동 자극에 대한 스파이크 기록',
         finding='H2의 활동전위를 직접 기록하였다.',
         mapping='MaleCNS type H2와 문헌의 H2를 세포 유형 수준에서 대응시킨다.',
         limitation='다른 개체의 유형 근거이며 MaleCNS 개체의 직접 측정이 아니다. 역치·불응기는 옮겨오지 않는다.'),
    dict(type_name='DM1_lPN', activity='spiking',
         evidence_level='recorded_type_transfer', species='Drosophila melanogaster',
         preparation='성체 2–10일; DM1 투사뉴런 전세포 기록', sex='female',
         source='https://doi.org/10.1523/JNEUROSCI.0764-09.2009',
         locator='Methods: NP5221-Gal4 DM1 PNs; Fig. 5B',
         finding='DM1 투사뉴런에서 실제 활동전위를 기록하였다.',
         mapping='DM1 uniglomerular PN을 MaleCNS DM1_lPN에 대응시키는 해부학적 추론이다.',
         limitation='암컷의 DM1 유형 근거를 수컷 대응 유형으로 전이한다. 모든 후각 투사뉴런으로 확장하지 않는다.'),
    *[dict(type_name=name, activity='graded',
           evidence_level='recorded_type_transfer', species='Drosophila melanogaster',
           preparation='성체 수컷; 생체 내 HS 전세포 기록', sex='male',
           source='https://doi.org/10.1038/s41467-024-53173-w',
           locator='Introduction; Fig. 5B–D; Methods: male flies',
           finding='HS의 운동 반응은 연속 막전위 변화로 다룬다. HSN/HSE/HSS 기록을 구별한다.',
           mapping=f'MaleCNS type {name}를 문헌의 동명 HS 유형에만 대응시킨다.',
           limitation='주된 연속 신호를 근사한다. 모든 조건에서 재생성 사건이 불가능하다는 주장이 아니다. 전기 시냅스는 추가하지 않는다.')
      for name in ('HSN', 'HSE', 'HSS')],
    dict(type_name='R1-R6', activity='graded',
         evidence_level='recorded_family_transfer', species='Drosophila melanogaster',
         preparation='생체 내 R1–R6 광수용체 세포 내 전압 기록', sex='이 근거집에서는 성별을 확정하지 않음',
         source='https://doi.org/10.3791/54142',
         locator='Introduction; Representative Results: R1–R6 voltage responses',
         finding='R1–R6의 빛 반응을 연속 전압 신호로 기록한다.',
         mapping='MaleCNS의 통합 라벨 R1-R6에만 적용한다. R7/R8에는 전파하지 않는다.',
         limitation='기록된 세포군의 주된 신호 방식을 전이한다. 개별 R1–R6의 매개변수 차이는 보존하지 못한다.'),
]


def assign_activity_modes(frame, evidence, unknown_mode='spiking'):
    if unknown_mode not in ('graded', 'spiking'):
        raise ValueError('미확인 뉴런의 기본값은 graded 또는 spiking이어야 한다.')
    if not frame.index.is_unique or 'type' not in frame:
        raise ValueError('고유 bodyId 인덱스와 type 열이 필요하다.')

    lookup = {}
    for entry in evidence:
        name = entry['type_name']
        if not isinstance(name, str) or not name or name != name.strip() or name in lookup:
            raise ValueError(f'중복 또는 잘못된 근거 유형: {name}')
        if entry['activity'] not in ('graded', 'spiking'):
            raise ValueError(f'잘못된 신호 방식: {name}')
        for key in ('source', 'locator', 'finding', 'mapping', 'limitation',
                    'evidence_level', 'species', 'preparation', 'sex'):
            if not isinstance(entry.get(key), str) or not entry[key].strip():
                raise ValueError(f'근거 필드 누락: {name}, {key}')
        if not entry['source'].startswith('https://'):
            raise ValueError(f'근거 출처에는 HTTPS 링크가 필요하다: {name}')
        lookup[name] = entry

    types = frame['type'].fillna('').astype(str).str.strip()

    # 근거 코드: 0은 미확인, 1은 연속 신호 근거, 2는 스파이크 근거이다.
    codes = types.map({
        k: 2 if v['activity'] == 'spiking' else 1
        for k, v in lookup.items()
    })
    codes = codes.fillna(0).to_numpy(dtype=np.uint8)

    records = pd.DataFrame({
        'bodyId': frame.index.to_numpy(),
        'type': types.to_numpy(),
        'evidence_code': codes,
    })
    records['evidence_activity'] = np.take(['unknown', 'graded', 'spiking'], codes)

    # 미확인 유형에만 기본값을 적용하고, 문헌상 연속 유형은 그대로 유지한다.
    records['assigned_activity'] = np.where(
        codes == 0, unknown_mode,
        np.where(codes == 2, 'spiking', 'graded'),
    )

    for key in ('source', 'locator', 'finding', 'mapping', 'limitation',
                'evidence_level', 'species', 'preparation', 'sex'):
        records[key] = types.map({
            k: v[key] for k, v in lookup.items()
        }).fillna('').to_numpy()

    records.loc[codes == 0, 'limitation'] = (
        f'이 근거집에서 미확인; {unknown_mode} 동역학을 가정한다. 직접 측정 근거가 아니다.'
    )
    records.loc[codes == 0, 'evidence_level'] = 'unknown_default'

    by_type = records.groupby(
        ['type', 'evidence_activity', 'assigned_activity'], dropna=False
    ).size()

    summary = dict(
        neurons=len(frame),
        evidence_supported_neurons=int((codes != 0).sum()),
        evidence_spiking_neurons=int((codes == 2).sum()),
        evidence_graded_neurons=int((codes == 1).sum()),
        unknown_default_neurons=int((codes == 0).sum()),
        evidence_fraction=float((codes != 0).mean()),
        unmatched_evidence_types=sorted(set(lookup) - set(types)),
        unknown_default=unknown_mode,
    )

    # 문헌 근거 코드와 실제 실행할 동역학 코드를 구분한다.
    # 미확인 뉴런의 evidence_code는 0으로 남지만, 실행 코드는 기본값을 따른다.
    assigned_codes = np.where(
        records['assigned_activity'].to_numpy() == 'spiking', 2, 1
    ).astype(np.uint8)

    return assigned_codes, records, by_type.reset_index(name='neurons'), summary


ACTIVITY_EVIDENCE.extend([
    dict(type_name=name,activity='graded',evidence_level='visual_model_assumption',
         species='Drosophila melanogaster',preparation='model transfer',sex='not specified',
         source='https://doi.org/10.1038/s41586-024-07939-3',locator='visual network graded model',
         finding='Use graded visual dynamics for the lamina proxy.',
         mapping='Exact MaleCNS type label',limitation='Model choice, not recording from this specimen.')
    for name in ['L1','L2','L3','L5']])
dynamics_ids, activity_annotations, activity_types, activity_coverage = assign_activity_modes(
    neurons, ACTIVITY_EVIDENCE, CFG['unknown_activity']
)

# graded_control에서는 배정 결과와 관계없이 모든 뉴런에 연속 동역학을 적용한다.
activity_annotations['effective_activity'] = np.where(
    (dynamics_ids == 2) & (CFG['neuron_mode'] == 'mixed'), 'spiking', 'graded'
)
activity_coverage['effective_spiking_neurons'] = int(
    ((dynamics_ids == 2) & (CFG['neuron_mode'] == 'mixed')).sum()
)
activity_coverage['neuron_mode'] = CFG['neuron_mode']

dynamics_spec = dict(
    version=EVIDENCE_VERSION,
    mode=CFG['neuron_mode'],
    unknown_activity=CFG['unknown_activity'],
    evidence=ACTIVITY_EVIDENCE,
    state_layout=['membrane', 'synaptic_signal', 'adaptation', 'refractory_ticks'],
    time_unit='simulation_tick_not_ms',
    parameters='engineering_assumptions',
)

dynamics_json = json.dumps(
    dynamics_spec, ensure_ascii=False, sort_keys=True, indent=2
)
dynamics_sha256 = hashlib.sha256(dynamics_json.encode('utf-8')).hexdigest()

activity_annotations.to_csv(RUN_DIR/'neuron_activity_annotations.csv', index=False)
activity_types.to_csv(RUN_DIR/'activity_types.csv', index=False)
(RUN_DIR/'dynamics_evidence.json').write_bytes(dynamics_json.encode('utf-8'))
(RUN_DIR/'activity_coverage.json').write_text(
    json.dumps(activity_coverage, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('신경 활동 근거 범위:', json.dumps(activity_coverage, ensure_ascii=False, indent=2))
print(activity_types.loc[activity_types.evidence_activity != 'unknown'].to_string(index=False))

if activity_coverage['unmatched_evidence_types']:
    print(
        '주의: 원본 type과 일치하지 않아 적용되지 않은 근거:',
        activity_coverage['unmatched_evidence_types'],
    )

if CFG['neuron_mode'] == 'mixed' and not np.any(dynamics_ids == 2):
    raise ValueError('스파이크 동역학이 배정된 뉴런이 없다. 유형 배정과 기본값을 확인하세요.')

print(
    f"미확인 유형의 배정 기본값: {CFG['unknown_activity']}. "
    '이는 문헌상 신호 방식이 확인되었다는 뜻이 아니다.'
)
print('graded_control 모드에서는 이 배정과 관계없이 모든 뉴런이 연속 동역학으로 실행된다.')

## 5. Model and direct copy objective

The model still reads **only the 67 annotated right-foreleg motors in the current release**
(the actual count is checked from the downloaded annotations). `decode_motor` simply
exposes the existing head for the cached-activity control. The full model remains image-only.
Its recurrent state is never carried from one training example to another in this notebook.

Contact loss averages positive and negative contact errors separately for each example.
Both recalls are reported, since an always-down pen can have a deceptively small unbalanced BCE.
Template NLL and nearest-template accuracy measure pen outputs, not a separate classifier.

In [ ]:
class SparseEdges(torch.autograd.Function):
    @staticmethod
    def forward(ctx, h, weights, rowptr, col, dst, block):
        matrix = torch.sparse_csr_tensor(rowptr, col, weights, size=(h.shape[1], h.shape[1]),
                                         device=h.device, dtype=h.dtype)
        ctx.save_for_backward(h, weights, rowptr, col, dst)
        ctx.block = block
        return torch.sparse.mm(matrix, h.T.contiguous()).T.contiguous()

    @staticmethod
    def backward(ctx, grad_output):
        h, weights, rowptr, col, dst = ctx.saved_tensors
        grad_h = torch.zeros_like(h) if ctx.needs_input_grad[0] else None
        grad_w = torch.empty_like(weights) if ctx.needs_input_grad[1] else None
        for lo in range(0, col.numel(), ctx.block):
            hi = min(col.numel(), lo+ctx.block)
            sources, targets = col[lo:hi], dst[lo:hi]
            g = grad_output.index_select(1, targets)
            if grad_w is not None:
                grad_w[lo:hi] = (g*h.index_select(1, sources)).sum(0)
            if grad_h is not None:
                grad_h.index_add_(1, sources, g*weights[lo:hi])
        return grad_h, grad_w, None, None, None, None

class SurrogateSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, voltage_gap, slope):
        ctx.save_for_backward(voltage_gap)
        ctx.slope = slope
        return (voltage_gap >= 0).to(voltage_gap.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        (gap,) = ctx.saved_tensors
        # 순전파는 이산 사건이며, 역전파에만 빠른 시그모이드의 대리 미분을 쓴다.
        return grad_output / (1 + ctx.slope * gap.abs()).square(), None


def mixed_neuron_tick(v, signal, slow, refractory, current, alpha, threshold,
                      adaptation, spiking_mask, reset_voltage, refractory_ticks, surrogate_slope):
    candidate = v + alpha[0] * (-v + current - adaptation * slow)
    blocked = spiking_mask & (refractory > 0)
    candidate = torch.where(blocked, torch.full_like(candidate, reset_voltage), candidate)
    spikes = SurrogateSpike.apply(candidate - threshold, surrogate_slope)
    spikes = spikes * (spiking_mask & ~blocked).to(candidate.dtype)
    graded = torch.relu(torch.tanh(candidate - threshold))
    release = torch.where(spiking_mask, spikes, graded)
    # 리셋·불응기 사건의 이산 결정에는 별도 대리 미분을 적용하지 않는다.
    fired = spikes.detach() > 0
    v = torch.where(fired, torch.full_like(candidate, reset_voltage), candidate)
    refractory = torch.where(fired, torch.full_like(refractory, float(refractory_ticks)),
                             torch.clamp(refractory - 1, min=0))
    refractory = torch.where(spiking_mask, refractory, torch.zeros_like(refractory)).detach()
    signal = signal + alpha[1] * (release - signal)
    slow = slow + alpha[2] * (release - slow)
    return v, signal, slow, refractory, spikes


def validate_dynamics_config(cfg):
    if cfg['neuron_mode'] not in ('mixed', 'graded_control'):
        raise ValueError('neuron_mode은 mixed 또는 graded_control이어야 한다.')
    if cfg['unknown_activity'] not in ('graded', 'spiking'):
        raise ValueError('미확인 뉴런의 기본값은 graded 또는 spiking이어야 한다.')
    for key in ('spike_threshold_center', 'spike_threshold_radius', 'spike_reset', 'surrogate_slope'):
        if not math.isfinite(cfg[key]):
            raise ValueError(f'유한한 동역학 설정이 필요하다: {key}')
    if (cfg['spike_threshold_radius'] < 0 or cfg['surrogate_slope'] <= 0
            or cfg['spike_threshold_center'] - cfg['spike_threshold_radius'] <= max(0, cfg['spike_reset'])):
        raise ValueError('스파이크 역치는 항상 0과 리셋 전압보다 높아야 한다.')
    if (type(cfg['refractory_ticks']) is not int or cfg['refractory_ticks'] < 0
            or type(cfg['brain_steps']) is not int or cfg['brain_steps'] < 1):
        raise ValueError('불응기는 0 이상의 정수, 토큰당 tick 수는 양의 정수여야 한다.')


class VisualPenLM(nn.Module):
    def __init__(self, cfg, rowptr, src, counts, edge_sign, inputs, outputs, type_ids, dynamics_ids, uv, lamina_nodes, device):
        super().__init__()
        validate_dynamics_config(cfg)
        self.tbptt = cfg['tbptt']
        self.reset_voltage = cfg['spike_reset']
        self.refractory_ticks = cfg['refractory_ticks']
        self.surrogate_slope = cfg['surrogate_slope']
        self.spike_threshold_center = cfg['spike_threshold_center']
        self.spike_threshold_radius = cfg['spike_threshold_radius']
        codes = np.asarray(dynamics_ids)
        if (codes.shape != (cfg['n_neurons'],) or codes.dtype != np.uint8
                or not np.isin(codes, [0, 1, 2]).all()):
            raise ValueError('뉴런 순서에 맞는 uint8 활동 코드 0/1/2가 필요하다.')
        mask = (codes == 2) & (cfg['neuron_mode'] == 'mixed')
        self.register_buffer('spiking_mask', torch.tensor(mask, dtype=torch.bool, device=device), persistent=False)
        self.n, self.k = cfg['n_neurons'], cfg['brain_steps']
        self.block, self.checkpoint_tokens = cfg['edge_block'], cfg['checkpoint_tokens']
        for name, value in [('rowptr', rowptr), ('src', src), ('inputs', inputs),
                            ('outputs', outputs), ('type_ids', type_ids)]:
            self.register_buffer(name, torch.tensor(np.asarray(value), dtype=torch.long,
                                                   device=device), persistent=False)
        assert len(type_ids) == self.n and min(type_ids) >= 0
        self.register_buffer('dst', torch.repeat_interleave(
            torch.arange(self.n, device=device), self.rowptr.diff()), persistent=False)
        sign = torch.tensor(np.asarray(edge_sign), dtype=torch.float32, device=device)
        self.register_buffer('edge_sign', sign, persistent=False)
        c = torch.tensor(np.asarray(counts), dtype=torch.float32, device=device).log1p()
        incoming = torch.zeros(self.n, device=device).index_add_(0, self.dst, c)
        self.register_buffer('prior', c / incoming[self.dst].clamp_min(1e-8), persistent=False)
        raw = torch.where(sign != 0, torch.full_like(sign, math.log(0.2 / 0.8)),
                          torch.where(torch.rand_like(sign) < 0.5, -0.2554128, 0.2554128))
        self.theta = nn.Parameter(raw)
        ntypes = int(max(type_ids)) + 1
        low, high, initial = [torch.tensor(cfg[key], device=device)[:, None]
                              for key in ['tau_min', 'tau_max', 'tau_initial']]
        assert torch.all((low > 0) & (initial > low) & (initial < high))
        self.register_buffer('tau_low', low)
        self.register_buffer('tau_high', high)
        fraction = (initial - low) / (high - low)
        self.raw_tau = nn.Parameter(torch.logit(fraction).expand(3, ntypes).clone())
        self.threshold = nn.Parameter(torch.zeros(ntypes, device=device))
        self.raw_adaptation = nn.Parameter(torch.full((ntypes,), math.log(0.1 / 0.9), device=device))
        self.cfg = dict(cfg)
        self.register_buffer('retina_grid', torch.tensor(uv*2-1,device=device).view(1,1,-1,2))
        self.register_buffer('lamina_nodes',torch.tensor(lamina_nodes,device=device,dtype=torch.long))
        self.register_buffer('joint_low',torch.tensor(cfg['joint_low'],device=device))
        self.register_buffer('joint_high',torch.tensor(cfg['joint_high'],device=device))
        self.motor_norm = nn.LayerNorm(len(outputs),eps=1e-6,elementwise_affine=False,device=device)
        self.pen_head = nn.Linear(len(outputs),cfg['pen_steps']*3,bias=True,device=device)
        nn.init.normal_(self.pen_head.weight,std=0.01)
        nn.init.zeros_(self.pen_head.bias)

    def effective_weights(self):
        magnitude = 0.25 + 3.75 * self.theta.sigmoid()
        signed = torch.where(self.edge_sign != 0, self.edge_sign * magnitude,
                             4 * self.theta.tanh())
        return self.prior * signed

    def dynamics(self):
        tau = self.tau_low + (self.tau_high - self.tau_low) * self.raw_tau.sigmoid()
        alpha = -torch.expm1(-1 / tau[:, self.type_ids])
        offset = self.threshold.tanh()[self.type_ids]
        threshold = torch.where(self.spiking_mask,
            self.spike_threshold_center + self.spike_threshold_radius * offset, 0.5 * offset)
        adaptation = self.raw_adaptation.sigmoid()[self.type_ids]
        return alpha, threshold, adaptation

    def initial_state(self, batch):
        return (self.theta.new_zeros(4,batch,self.n),
                self.theta.new_zeros(batch,len(self.inputs)))

    def sample_retina(self, images):
        if images.ndim != 4 or images.shape[1] != 1 or not images.is_floating_point():
            raise ValueError('Only floating-point grayscale images [B,1,H,W] enter the model')
        # Linearize the screen before spatial sampling.
        linear = torch.where(images <= .04045, images/12.92, ((images+.055)/1.055).pow(2.4))
        blur = self.cfg['retinal_blur']
        linear = F.avg_pool2d(linear,kernel_size=blur,stride=1,padding=blur//2)
        return F.grid_sample(linear,self.retina_grid.expand(len(images),-1,-1,-1),
            mode='bilinear',align_corners=True).reshape(len(images),-1)

    def external_drive(self, luminance):
        value = self.cfg['retinal_gain']*luminance/(self.cfg['retinal_half_saturation']+luminance)
        return luminance.new_zeros(len(luminance),self.n).index_add(1,self.inputs,value)

    def advance(self, image, neural, retina, weights, alpha, threshold, adaptation):
        v,signal,slow,refractory = neural.unbind(0)
        luminance = self.sample_retina(image)
        eye_alpha = 1-math.exp(-1/self.cfg['retina_tau_ticks'])
        intrinsic = torch.zeros_like(v)
        intrinsic[:,self.lamina_nodes] = self.cfg['lamina_rest']
        for _ in range(self.k):
            retina = retina + eye_alpha*(luminance-retina)
            message = SparseEdges.apply(signal,weights,self.rowptr,self.src,self.dst,self.block)
            v,signal,slow,refractory,_ = mixed_neuron_tick(v,signal,slow,refractory,
                message+self.external_drive(retina)+intrinsic,alpha,threshold,adaptation,
                self.spiking_mask,self.reset_voltage,self.refractory_ticks,self.surrogate_slope)
        return torch.stack((v,signal,slow,refractory)),retina

    def decode_pen(self, neural):
        return self.decode_motor(neural[1].index_select(1, self.outputs))

    def decode_motor(self, motor_signal):
        motor = self.motor_norm(motor_signal)
        raw = self.pen_head(motor).reshape(len(motor),self.cfg['pen_steps'],3)
        angles = self.joint_low+(self.joint_high-self.joint_low)*raw[...,:2].sigmoid()
        xy = forward_kinematics(angles,self.cfg['link_lengths'])
        return xy,raw[...,2],angles

    def forward(self, images, state=None):
        if images.ndim != 5 or images.shape[2] != 1 or images.shape[1] == 0 or not images.is_floating_point():
            raise ValueError('Expected float image sequence [B,T,1,H,W], never character IDs')
        neural,retina = self.initial_state(len(images)) if state is None else state
        assert neural.shape == (4,len(images),self.n)
        assert retina.shape == (len(images),len(self.inputs))
        weights = self.effective_weights(); alpha,threshold,adaptation = self.dynamics()
        positions,contacts,angles = [],[],[]
        for t in range(images.shape[1]):
            args = (images[:,t],neural,retina,weights,alpha,threshold,adaptation)
            if self.training and self.checkpoint_tokens and torch.is_grad_enabled():
                neural,retina = checkpoint(self.advance,*args,use_reentrant=False,preserve_rng_state=False)
            else:
                neural,retina = self.advance(*args)
            xy,ink,q = self.decode_pen(neural)
            positions.append(xy); contacts.append(ink); angles.append(q)
        return (torch.stack(positions,1),torch.stack(contacts,1),torch.stack(angles,1)),(neural,retina)

def make_model():
    seed_all(CFG['seed'])
    return VisualPenLM(CFG,crow,src,counts,edge_sign,input_nodes,output_nodes,type_ids,
        dynamics_ids,uv,lamina_nodes,device)

def detach_state(state):
    return tuple(x.detach() for x in state)

def template_logits(xy, ink_logits):
    # B*T by C only; no B*T*C*S intermediate allocation.
    a=xy.flatten(-2); b=target_trace[...,:2].flatten(-2)
    distance=(a.square().mean(-1,keepdim=True)+b.square().mean(-1)
              -2*(a@b.T)/a.shape[-1]).clamp_min(0)
    ink=ink_logits.sigmoid(); template_ink=target_trace[...,2]
    distance=distance+CFG['ink_loss_weight']*(ink.square().mean(-1,keepdim=True)+
        template_ink.square().mean(-1)-2*(ink@template_ink.T)/ink.shape[-1]).clamp_min(0)
    return -distance/CFG['template_temperature']


class IndependentCopyPredictor(nn.Module):
    def __init__(self, core):
        super().__init__()
        self.core = core

    def forward(self, images):
        prediction, _ = self.core(images)
        # Only batch-first predictions may be gathered. Neural state is [4,B,N],
        # and gathering it on dimension 0 would corrupt the replica batch layout.
        return prediction

def make_parallel_predictor(model):
    wrapper = IndependentCopyPredictor(model)
    if len(GPU_IDS) > 1:
        return nn.DataParallel(wrapper, device_ids=GPU_IDS, output_device=GPU_IDS[0], dim=0)
    return wrapper

def fraction_true(mask):
    # Integer counts avoid float32 reduction returning 0.99999994 for an all-true mask.
    return int(torch.count_nonzero(mask))/mask.numel()

def example_losses(prediction, labels):
    xy, ink, _ = prediction
    target = target_trace[labels]
    pos = (xy - target[..., :2]).square().mean((-1, -2))
    down = target[..., 2]
    bce = F.binary_cross_entropy_with_logits(ink, down, reduction='none')
    down_error = (bce*down).sum(-1) / down.sum(-1).clamp_min(1)
    up_error = (bce*(1-down)).sum(-1) / (1-down).sum(-1).clamp_min(1)
    contact = 0.5*(down_error + up_error)
    loss = pos/POSITION_SCALE + CFG['ink_loss_weight']*contact
    return loss, pos, contact, bce.mean(-1)

def objective(prediction, labels):
    loss, pos, contact, raw_bce = example_losses(prediction, labels)
    with torch.no_grad():
        logits = template_logits(prediction[0], prediction[1])
        target_down = target_trace[labels, :, 2] >= .5
        predicted_down = prediction[1] >= 0
        metrics = dict(loss=float(loss.mean().detach()), tip_mse=float(pos.mean().detach()),
            relative_tip_mse=float(pos.mean().detach()/POSITION_SCALE),
            balanced_contact_bce=float(contact.mean().detach()),
            contact_bce=float(raw_bce.mean().detach()),
            template_nll=float(F.cross_entropy(logits.flatten(0, 1), labels.flatten())),
            char_accuracy=fraction_true(logits.argmax(-1) == labels),
            ink_recall=fraction_true(predicted_down[target_down]),
            penup_recall=fraction_true(~predicted_down[~target_down]))
    return loss.mean(), metrics

def as_sequence(prediction):
    return tuple(value[:, None] for value in prediction)

def gradient_norm(parameter):
    return 0.0 if parameter.grad is None else float(parameter.grad.detach().norm())

def gradient_report(model):
    return dict(edge_grad_norm=gradient_norm(model.theta),
        tau_grad_norm=gradient_norm(model.raw_tau),
        threshold_grad_norm=gradient_norm(model.threshold),
        adaptation_grad_norm=gradient_norm(model.raw_adaptation),
        head_weight_grad_norm=gradient_norm(model.pen_head.weight),
        head_bias_grad_norm=gradient_norm(model.pen_head.bias))

## 6. Numerical and task checks

The inherited sparse-gradient, retinal, motor, spiking/refractory and checkpoint tests run
on CPU and CUDA. Additional checks cover same-letter labels, balanced data, exact-trace loss,
and the always-down pen failure. These tests do not substitute for full-graph overfitting.

In [ ]:
def small_checks(test_device):
    seed_all(123)
    row=torch.tensor([0,1,2,4],device=test_device)
    col=torch.tensor([2,0,0,1],device=test_device)
    dst=torch.tensor([0,1,2,2],device=test_device)
    h=torch.randn(2,3,device=test_device,requires_grad=True)
    w=torch.randn(4,device=test_device,requires_grad=True)
    actual=SparseEdges.apply(h,w,row,col,dst,2)
    matrix=torch.zeros(3,3,device=test_device).index_put((dst,col),w)
    expected=h@matrix.T
    torch.testing.assert_close(actual,expected)
    grad=torch.randn_like(actual)
    ga=torch.autograd.grad(actual,(h,w),grad,retain_graph=True)
    ge=torch.autograd.grad(expected,(h,w),grad)
    for a,b in zip(ga,ge): torch.testing.assert_close(a,b)
    cfg=dict(CFG,n_neurons=6,brain_steps=8,neuron_mode='graded_control',lamina_rest=0.)
    model=VisualPenLM(cfg,np.array([0,0,0,1,2,3,4]),np.array([0,1,2,3]),
        np.ones(4),np.ones(4),np.array([0,1]),np.array([4,5]),np.arange(6),
        np.ones(6,dtype=np.uint8),np.array([[.2,.5],[.8,.5]],np.float32),
        np.array([],dtype=np.int64),test_device)
    image=torch.zeros(1,2,1,12,12,device=test_device)
    image[:,:,:,:,:6]=.8; image.requires_grad_(); model.eval()
    pred,state=model(image); pred2,state2=model(image.flip(-1))
    drive=model.external_drive(torch.ones(1,2,device=test_device))
    assert torch.count_nonzero(drive[:,2:])==0
    assert not torch.allclose(pred[0],pred2[0]), 'Images do not affect the pen'
    (pred[0].square().mean()+pred[1].square().mean()).backward()
    assert image.grad is not None and image.grad.abs().sum()>0
    assert model.theta.grad is not None and model.theta.grad.abs().sum()>0
    s1,s2=state[0].clone(),state2[0].clone()
    s1[1,:,model.outputs]=0; s2[1,:,model.outputs]=0
    torch.testing.assert_close(model.decode_pen(s1)[0],model.decode_pen(s2)[0])
    try: model(torch.zeros(1,2,dtype=torch.long,device=test_device))
    except ValueError: pass
    else: raise AssertionError('Model accepted character IDs')
    z=torch.zeros(1,1,device=test_device); mask=torch.ones_like(z,dtype=torch.bool)
    args=(torch.ones(3,1,device=test_device),torch.full_like(z,.25),z,mask,0.,2,5.)
    v,s,a,r,sp=mixed_neuron_tick(z,z,z,z,torch.ones_like(z),*args)
    assert sp.item()==1 and r.item()==2 and v.item()==0
    *_,blocked=mixed_neuron_tick(v,s,a,r,torch.ones_like(z),*args)
    assert blocked.item()==0
    model.zero_grad(set_to_none=True); model.train(); model.checkpoint_tokens=False
    plain,_=model(image.detach()); plain[0].sum().backward(); gp=model.theta.grad.clone()
    model.zero_grad(set_to_none=True); model.checkpoint_tokens=True
    checked,_=model(image.detach()); checked[0].sum().backward()
    torch.testing.assert_close(plain[0],checked[0]); torch.testing.assert_close(gp,model.theta.grad)
    print('PASS sparse, retinal, motor, refractory, checkpoint checks:',test_device)
small_checks(torch.device('cpu'))
if device.type=='cuda': small_checks(device)
with torch.no_grad():
    canonical_ink=torch.logit(target_trace[...,2].clamp(.00001,.99999))
    recognized=template_logits(target_trace[...,:2],canonical_ink).argmax(-1)
    assert torch.equal(recognized,torch.arange(len(chars),device=device)), 'Canonical traces must self-recognize'
print('PASS: every canonical pen trace self-recognizes')

In [ ]:
canonical_prediction = (target_trace[:, None, :, :2],
    torch.logit(target_trace[:, None, :, 2].clamp(1e-5, 1-1e-5)),
    target_joints[:, None])
canonical_labels = torch.arange(3, device=device)[:, None]
_, CANONICAL_METRICS = objective(canonical_prediction, canonical_labels)
assert CANONICAL_METRICS['char_accuracy'] == 1
assert CANONICAL_METRICS['relative_tip_mse'] < 1e-10
assert CANONICAL_METRICS['penup_recall'] == CANONICAL_METRICS['ink_recall'] == 1
always_down = (canonical_prediction[0], torch.full_like(canonical_prediction[1], 12.), canonical_prediction[2])
_, always_down_metrics = objective(always_down, canonical_labels)
assert always_down_metrics['penup_recall'] == 0
assert always_down_metrics['balanced_contact_bce'] > 5
for u in [1, 2, len(train_batches), len(train_batches)+1]:
    images, labels = copy_batch(u)
    torch.testing.assert_close(images[:, 0], glyph_images[labels[:, 0]])
    assert all(sorted(triple) == [0, 1, 2] for triple in labels[:, 0].reshape(-1, 3).tolist())
print('PASS ABC labels, balance, canonical objective and pen-up regression checks.')
print('Canonical reference (template NLL need not be zero):', CANONICAL_METRICS)

## 7. Full-graph propagation and gradient preflight

Retinal separability alone is insufficient. The table measures pairwise A/B/C distances at
both the retina and the motors, normalized motor features, active motors and feature rank.
Each latency setting starts from the same zero state. No setting is silently tuned from labels.
The three-letter comparisons are diagnostic uses of the overfit set, not validation claims.

The actual configured model also runs an Adam step and verifies image gradients, synaptic
gradients and motor-only causality. It checks the post-update pen response to all three
letters, not merely real versus blank input. Training starts from a freshly seeded model.

In [ ]:
@torch.no_grad()
def motor_statistics(model, frames):
    prediction, state = model(frames)
    motor = state[0][1].index_select(1, model.outputs)
    features = model.motor_norm(motor)
    centered = features - features.mean(0, keepdim=True)
    return dict(motor_min_pair_l2=float(torch.pdist(motor).min()),
        feature_min_pair_l2=float(torch.pdist(features).min()),
        feature_rank=int(torch.linalg.matrix_rank(centered, atol=1e-5)),
        active_motors=int((motor > 1e-8).any(0).sum()),
        motor_mean=float(motor.mean()),
        pen_min_pair_l2=float(torch.pdist(prediction[0][:, 0].flatten(1)).min()))

def full_preflight():
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        for gpu in GPU_IDS: torch.cuda.reset_peak_memory_stats(gpu)
    test = make_model()
    parallel_test = make_parallel_predictor(test)
    frames = glyph_images[:, None]
    with torch.no_grad():
        retinal = test.sample_retina(glyph_images)
        retinal_min = float(torch.pdist(retinal).min())
        assert retinal_min > 1e-5, 'ABC images are indistinguishable at the mapped retina.'
        latency = []
        configured_ticks = test.k
        for ticks in sorted({4, 8, 16, 32, 64, configured_ticks}):
            test.k = ticks
            latency.append(dict(ticks=ticks, **motor_statistics(test, frames)))
        test.k = configured_ticks
        configured = next(row for row in latency if row['ticks'] == configured_ticks)
        assert configured['motor_min_pair_l2'] > 1e-7, (
            'ABC motor signals coincide at configured ticks. Inspect the latency table and dynamics.')
        assert configured['feature_rank'] == 2, 'Three motor patterns need two independent contrasts.'
    print(pd.DataFrame(latency).to_string(index=False))
    images, labels = copy_batch(1)
    images = images.clone().requires_grad_(True)
    opt = torch.optim.Adam([
        {'params': [test.theta, test.raw_tau, test.threshold, test.raw_adaptation], 'lr': CFG['lr']},
        {'params': test.pen_head.parameters(), 'lr': CFG['joint_head_lr']}], foreach=False)
    start = time.perf_counter()
    prediction = parallel_test(images)
    with torch.no_grad():
        reference, _ = test(images.detach())
        for actual, expected in zip(prediction, reference):
            torch.testing.assert_close(actual, expected, atol=2e-5, rtol=2e-5)
        del reference
    loss, metrics = objective(prediction, labels)
    loss.backward()
    grads = gradient_report(test)
    image_grad_norm = float(images.grad.norm())
    assert image_grad_norm > 0 and math.isfinite(image_grad_norm), 'No usable image gradient.'
    assert grads['edge_grad_norm'] > 0 and all(math.isfinite(v) for v in grads.values())
    torch.nn.utils.clip_grad_norm_(test.parameters(), 1., error_if_nonfinite=True)
    opt.step(); test.eval()
    with torch.no_grad():
        after = motor_statistics(test, frames)
        assert after['pen_min_pair_l2'] > 1e-7, 'An optimizer step erased ABC output differences.'
        real, _ = test(frames)
        blank, _ = test(torch.zeros_like(frames))
        delta = float((real[0]-blank[0]).abs().max())
        assert delta > 1e-7
        # The entire forward result must follow a batch permutation with no lane identity.
        permutation = torch.tensor([2, 0, 1], device=device)
        permuted, _ = test(frames[permutation])
        for expected, actual in zip(real, permuted):
            torch.testing.assert_close(actual, expected[permutation], atol=2e-5, rtol=2e-5)
    for gpu in GPU_IDS: torch.cuda.synchronize(gpu)
    peaks = {str(gpu): torch.cuda.max_memory_allocated(gpu)/2**30 for gpu in GPU_IDS}
    result = dict(metrics=metrics, gradients=grads, image_grad_norm=image_grad_norm,
        retinal_min_pair_l2=retinal_min, configured=configured, latency=latency,
        after_optimizer_step=after, blank_pen_max_delta=delta,
        seconds=time.perf_counter()-start, gpu_peak_allocated_GiB=peaks,
        training_gpu_ids=GPU_IDS, global_batch=CFG['batch_size'])
    for gpu in GPU_IDS:
        assert peaks[str(gpu)] < CFG['vram_fraction']*torch.cuda.get_device_properties(gpu).total_memory/2**30
    del parallel_test, test, opt, prediction, loss, images, real, blank, permuted
    gc.collect()
    if device.type == 'cuda': torch.cuda.empty_cache()
    seed_all(CFG['seed'])
    return result

PREFLIGHT = full_preflight()
(RUN_DIR/'preflight.json').write_text(json.dumps(PREFLIGHT, indent=2), encoding='utf-8')
print('PASS full-graph ABC preflight:', PREFLIGHT['gradients'])

## 8. Fit criteria, interventions and baselines

Metrics are computed over the three unique canonical examples, each from a reset state.
Accuracy alone can pass with poor handwriting, so the stopping criteria also require
position error substantially below the mean-trace baseline and both contact recalls.

Interventions keep target labels A/B/C fixed: blank images, all-A images, cyclically permuted
images B/C/A and clamped motor readout. A working copier should follow the **presented**
permuted image (100% presented-image accuracy, 0% original-label accuracy). Constant
inputs and clamped motors cannot exceed 1/3 on this balanced test. We also compare
batch permutations to exclude state or lane leakage.

In [ ]:
@torch.no_grad()
def prediction_metrics(prediction, labels):
    _, result = objective(prediction, labels)
    losses, pos, contact, raw_bce = example_losses(prediction, labels)
    guesses = template_logits(prediction[0], prediction[1]).argmax(-1)
    per_class = {}
    for label, ch in enumerate(chars):
        mask = labels == label
        target_ink = target_trace[labels[mask], :, 2] >= .5
        predicted_ink = prediction[1][mask] >= 0
        per_class[ch] = dict(char_accuracy=fraction_true(guesses[mask] == label),
            tip_mse=float(pos[mask].mean()),
            relative_tip_mse=float(pos[mask].mean()/CLASS_POSITION_SCALE[label]),
            ink_recall=fraction_true(predicted_ink[target_ink]),
            penup_recall=fraction_true(~predicted_ink[~target_ink]))
    result['per_class'] = per_class
    result['predicted'] = ''.join(chars[i] for i in guesses.flatten().tolist())
    result['max_class_relative_mse'] = max(r['relative_tip_mse'] for r in per_class.values())
    result['min_class_ink_recall'] = min(r['ink_recall'] for r in per_class.values())
    result['min_class_penup_recall'] = min(r['penup_recall'] for r in per_class.values())
    return result

def fits_targets(metrics):
    return (metrics['char_accuracy'] == 1.0
        and metrics['relative_tip_mse'] <= CFG['fit_relative_mse']
        and metrics['max_class_relative_mse'] <= CFG['fit_max_class_relative_mse']
        and metrics['min_class_ink_recall'] >= CFG['fit_ink_recall']
        and metrics['min_class_penup_recall'] >= CFG['fit_penup_recall'])

@torch.no_grad()
def evaluate(model, interventions=True):
    previous = model.training; model.eval()
    labels = torch.arange(3, device=device)[:, None]
    frames = glyph_images[:, None]
    try:
        prediction, state = model(frames)
        real = prediction_metrics(prediction, labels)
        motor = state[0][1].index_select(1, model.outputs)
        real['motor_min_pair_l2'] = float(torch.pdist(motor).min())
        report = dict(real=real, target_fit_pass=fits_targets(real), unique_examples=3)
        if interventions:
            blind, _ = model(torch.zeros_like(frames))
            constant, _ = model(frames[:1].expand_as(frames))
            permutation = torch.tensor([1, 2, 0], device=device)
            permuted, _ = model(frames[permutation])
            hook = model.motor_norm.register_forward_pre_hook(
                lambda module, args: (torch.zeros_like(args[0]),))
            try:
                clamped, _ = model(frames)
            finally:
                hook.remove()
            report.update(blank=prediction_metrics(blind, labels),
                constant_A=prediction_metrics(constant, labels),
                shuffled_original_labels=prediction_metrics(permuted, labels),
                shuffled_presented_labels=prediction_metrics(permuted, labels[permutation]),
                motor_clamped=prediction_metrics(clamped, labels),
                blank_pen_max_delta=float((prediction[0]-blind[0]).abs().max()))
            report['order_max_delta'] = max(float((p-p0[permutation]).abs().max())
                for p, p0 in zip(permuted, prediction))
            chance = 1/3 + 1e-6
            report['vision_dependence_pass'] = bool(
                report['blank']['char_accuracy'] <= chance
                and report['constant_A']['char_accuracy'] <= chance
                and report['motor_clamped']['char_accuracy'] <= chance
                and report['shuffled_original_labels']['char_accuracy'] <= chance
                and report['shuffled_presented_labels']['char_accuracy'] == 1.0
                and report['blank_pen_max_delta'] > 1e-5
                and report['order_max_delta'] < 1e-4)
            report['pass'] = bool(report['target_fit_pass'] and report['vision_dependence_pass'])
        return report
    finally:
        model.train(previous)

mean_ink = target_trace[..., 2].mean(0).clamp(1e-5, 1-1e-5)
baseline_prediction = (mean_xy[None, None].expand(3, 1, -1, -1),
    torch.logit(mean_ink)[None, None].expand(3, 1, -1),
    inverse_kinematics(mean_xy, CFG['link_lengths'])[None, None].expand(3, 1, -1, -1))
MEAN_TRACE_BASELINE = prediction_metrics(baseline_prediction, canonical_labels)
assert abs(MEAN_TRACE_BASELINE['relative_tip_mse']-1) < 1e-5
BASELINES = dict(mean_trace=MEAN_TRACE_BASELINE, canonical=CANONICAL_METRICS,
    uniform_char_accuracy=1/3, uniform_template_nll=math.log(3))
(RUN_DIR/'baselines.json').write_text(json.dumps(BASELINES, indent=2), encoding='utf-8')
print('Mean-trace baseline:', {k: v for k, v in MEAN_TRACE_BASELINE.items() if k != 'per_class'})

## 9. Self-contained assets and resumable checkpoints

Runs save the exact dataset, graph arrays, font/trace fingerprints, configuration and
separate readout/full-training metrics. Checkpoints use `flylm3_abc_copy_v1`; resume only
this experiment, not a Tiny Shakespeare checkpoint. The display/replay below loads this
model directly; the original language-generation notebook is not an ABC copy evaluator.

`best.pt` includes the best model across both stages, with its phase recorded.
`latest.pt` resumes the end-to-end stage and its optimizer from the completed update.
Selecting a readout-stage checkpoint as best does not establish that circuit training helped.

In [ ]:
def save_array_assets():
    graph_dir=RUN_DIR/'graph'; graph_dir.mkdir(exist_ok=True)
    arrays=dict(body_ids=body_ids,crow=crow,src=src,counts=counts,edge_sign=edge_sign,
        input_nodes=input_nodes,output_nodes=output_nodes,type_ids=type_ids,dynamics_ids=dynamics_ids,
        uv=uv,lamina_nodes=lamina_nodes)
    fingerprints={}
    for key,value in arrays.items():
        np.save(graph_dir/(key+'.npy'),value)
        fingerprints[key]=sha256_file(graph_dir/(key+'.npy'))
    torch.save(dict(images=glyph_images_cpu,trace=target_trace_cpu,joints=target_joints_cpu,chars=chars),
        RUN_DIR/'visual_pen_assets.pt')
    spec=dict(format='flylm3_abc_copy_v1',cfg=CFG,data=DATA_SPEC,font=FONT_SPEC,
        graph=GRAPH_SPEC,sources=source_manifest,dynamics=dynamics_spec,versions=VERSIONS)
    (RUN_DIR/'experiment.json').write_text(json.dumps(spec,ensure_ascii=False,indent=2),encoding='utf-8')
    semantic=dict(cfg=CFG,data=DATA_SPEC,dynamics=dynamics_spec,arrays=fingerprints,
        image_sha=hashlib.sha256(glyph_images_cpu.numpy().tobytes()).hexdigest(),
        trace_sha=hashlib.sha256(target_trace_cpu.numpy().tobytes()).hexdigest())
    (RUN_DIR/'asset_manifest.json').write_text(json.dumps(semantic,indent=2),encoding='utf-8')
    return hashlib.sha256(json.dumps(semantic,sort_keys=True).encode()).hexdigest()


FORMAT = 'flylm3_abc_copy_v1'
EXPERIMENT_ID = save_array_assets()

def make_optimizer(model):
    return torch.optim.Adam([
        {'params': [model.theta, model.raw_tau, model.threshold, model.raw_adaptation], 'lr': CFG['lr']},
        {'params': list(model.pen_head.parameters()), 'lr': CFG['joint_head_lr']}], foreach=False)

def save_checkpoint(path, model, optimizer, update, best, phase):
    payload = dict(format=FORMAT, identity=EXPERIMENT_ID, model=model.state_dict(),
        optimizer=optimizer.state_dict(), update=update, best=best, phase=phase,
        torch_rng=torch.get_rng_state(),
        cuda_rng=torch.cuda.get_rng_state_all() if device.type == 'cuda' else None)
    partial = path.with_suffix('.part')
    torch.save(payload, partial); partial.replace(path)

def restore_checkpoint(path, model, optimizer=None):
    saved = torch.load(path, map_location='cpu', weights_only=False)
    if saved.get('format') != FORMAT or saved.get('identity') != EXPERIMENT_ID:
        raise ValueError('ABC checkpoint/config/data/graph/assets mismatch.')
    model.load_state_dict(saved['model'])
    if optimizer is not None:
        optimizer.load_state_dict(saved['optimizer'])
        torch.set_rng_state(saved['torch_rng'])
        if device.type == 'cuda' and saved['cuda_rng'] is not None:
            torch.cuda.set_rng_state_all(saved['cuda_rng'])
    return dict(update=int(saved['update']), best=float(saved['best']), phase=saved['phase'])

def write_metric(row):
    with (RUN_DIR/'metrics.jsonl').open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row)+'\n')

## 10. Stage 1 — can a head learn from the untrained circuit?

Cache one raw motor vector per canonical image from the full, freshly initialized model.
Fit only the original pen head; no labels or pixels become features. This inexpensive stage
distinguishes an unreadable motor code from difficulty optimizing the recurrent circuit.
The final readout test always reruns the actual model from images and applies interventions.

Set `RESUME_FROM` to an ABC checkpoint to skip this stage and restore end-to-end training.
Configuration, data and graph fingerprints must match. Training order is a deterministic
function of the update number, so no hidden stream state is required.

In [ ]:
RESUME_FROM = None  # Path('/kaggle/input/my-abc-run/latest.pt')
model = make_model()
parallel_model = make_parallel_predictor(model)
optimizer = make_optimizer(model)
start_update = 0
best = float('inf')
INITIAL = evaluate(model)
print('Initial:', INITIAL['real'])
READOUT = None
if RESUME_FROM is None:
    with torch.no_grad():
        model.eval()
        _, state = model(glyph_images[:, None])
        cached_motor = state[0][1].index_select(1, model.outputs).detach().clone()
        del state
    head_optimizer = torch.optim.Adam(model.pen_head.parameters(), lr=CFG['head_lr'], foreach=False)
    for head_update in range(1, CFG['head_updates']+1):
        model.train(); model.zero_grad(set_to_none=True)
        _, labels = copy_batch(head_update)
        prediction = as_sequence(model.decode_motor(cached_motor[labels[:, 0]]))
        loss, metrics = objective(prediction, labels)
        loss.backward()
        assert model.theta.grad is None, 'Readout control must not train circuit edges.'
        torch.nn.utils.clip_grad_norm_(model.pen_head.parameters(), 1., error_if_nonfinite=True)
        head_optimizer.step()
        if head_update == 1 or head_update % 50 == 0 or head_update == CFG['head_updates']:
            row = dict(phase='frozen_circuit_readout', update=head_update, **metrics)
            print(row); write_metric(row)
    READOUT = evaluate(model)
    best = READOUT['real']['loss']
    save_checkpoint(RUN_DIR/'best.pt', model, optimizer, 0, best, 'frozen_circuit_readout')
    save_checkpoint(RUN_DIR/'latest.pt', model, optimizer, 0, best, 'frozen_circuit_readout')
    del head_optimizer, cached_motor, prediction, loss
    model.zero_grad(set_to_none=True)
    print('Frozen-circuit readout:', READOUT['real']['predicted'],
          'relative MSE:', READOUT['real']['relative_tip_mse'], 'PASS:', READOUT['pass'])
else:
    resumed = restore_checkpoint(Path(RESUME_FROM), model, optimizer)
    start_update = resumed['update']
    # The supplied checkpoint is the best available candidate in this new run directory.
    restored_evaluation = evaluate(model)
    best = restored_evaluation['real']['loss']
    save_checkpoint(RUN_DIR/'best.pt', model, optimizer, start_update, best, resumed['phase'])
    save_checkpoint(RUN_DIR/'latest.pt', model, optimizer, start_update, best, resumed['phase'])
    print('Resumed:', resumed)
(RUN_DIR/'stage1.json').write_text(json.dumps(dict(initial=INITIAL, frozen_readout=READOUT,
    resumed_from=None if RESUME_FROM is None else str(RESUME_FROM)), indent=2), encoding='utf-8')

## 11. Stage 2 — end-to-end fitting and collapse monitoring

Every update recomputes all configured neural ticks for three reset images **per GPU**. Circuit and
head gradients are logged separately **before clipping**, and each group is clipped separately.
No cached motor vectors are used here. A low aggregate loss alone cannot stop this run.

The saved best checkpoint can remain the frozen-circuit readout if end-to-end updates
make the spike-based representation worse. The final report explicitly compares the phases.

In [ ]:
start_time = time.perf_counter()
last_update = start_update
consecutive_passes = 0
stop_reason = 'update_budget_reached'
circuit_parameters = [model.theta, model.raw_tau, model.threshold, model.raw_adaptation]
for update in range(start_update+1, CFG['train_updates']+1):
    model.train()
    images, labels = copy_batch(update)
    optimizer.zero_grad(set_to_none=True)
    prediction = parallel_model(images)  # Replica predictions only; no carried/gathered state.
    loss, metrics = objective(prediction, labels)
    if not torch.isfinite(loss): raise FloatingPointError(f'Nonfinite loss at update {update}')
    loss.backward()
    grads = gradient_report(model)
    circuit_norm = float(torch.nn.utils.clip_grad_norm_(circuit_parameters, 1., error_if_nonfinite=True))
    head_norm = float(torch.nn.utils.clip_grad_norm_(model.pen_head.parameters(), 1., error_if_nonfinite=True))
    optimizer.step()
    last_update = update
    elapsed = time.perf_counter()-start_time
    completed = update-start_update
    row = dict(phase='end_to_end', update=update, **metrics, **grads,
        circuit_grad_norm=circuit_norm, head_grad_norm=head_norm,
        characters_per_second=CFG['batch_size']*completed/elapsed, elapsed_seconds=elapsed,
        eta_seconds=elapsed/completed*(CFG['train_updates']-update))
    # Drop the entire recurrent/autograd graph before evaluation or serialization.
    del loss, prediction
    if update == start_update+1 or update % CFG['log_every'] == 0:
        print(row)
    if update % CFG['eval_every'] == 0 or update == CFG['train_updates']:
        validation = evaluate(model)
        row['evaluation'] = validation
        print('Evaluation:', {k: v for k, v in validation['real'].items() if k != 'per_class'},
              'PASS:', validation['pass'])
        if validation['real']['loss'] < best:
            best = validation['real']['loss']
            save_checkpoint(RUN_DIR/'best.pt', model, optimizer, update, best, 'end_to_end')
        consecutive_passes = consecutive_passes+1 if validation['pass'] else 0
    write_metric(row)
    if update % CFG['save_every'] == 0:
        save_checkpoint(RUN_DIR/'latest.pt', model, optimizer, update, best, 'end_to_end')
    if consecutive_passes >= CFG['early_stop_evals']:
        stop_reason = 'fit_and_interventions_passed_repeatedly'
        break

save_checkpoint(RUN_DIR/'latest.pt', model, optimizer, last_update, best,
                'end_to_end' if last_update > start_update else 'resumed_without_updates')
LATEST_EVALUATION = evaluate(model)
best_info = restore_checkpoint(RUN_DIR/'best.pt', model)
BEST_EVALUATION = evaluate(model)
TRAINING_SUMMARY = dict(last_update=last_update, stop_reason=stop_reason,
    best_checkpoint=best_info, latest=LATEST_EVALUATION, best=BEST_EVALUATION)
(RUN_DIR/'training_summary.json').write_text(json.dumps(TRAINING_SUMMARY, indent=2), encoding='utf-8')
print('Best checkpoint:', best_info, 'overall PASS:', BEST_EVALUATION['pass'])

## 12. Inspect the actual pen drawings and perturbations

This displays **predicted paths and their own rasterizations**, never replacement font
glyphs selected by a classifier. The best checkpoint is already loaded. A fourth panel
shows the mean-trace baseline. Saved PNG/NPZ files permit inspection without this kernel.

In [ ]:
@torch.no_grad()
def predict_traces(model, frames):
    previous = model.training; model.eval()
    try:
        prediction, _ = model(frames)
        xy, ink, joints = prediction
        traces = torch.cat([xy[:, 0], ink[:, 0].sigmoid()[..., None]], -1).cpu().numpy()
        return traces, joints[:, 0].cpu().numpy()
    finally:
        model.train(previous)

traces, joints = predict_traces(model, glyph_images[:, None])
np.savez(RUN_DIR/'abc_predictions.npz', traces=traces, joints=joints,
         target=target_trace_cpu.numpy(), chars=np.array(chars))
fig, axes = plt.subplots(3, 4, figsize=(11, 8))
for i, ch in enumerate(chars):
    axes[0, i].imshow(glyph_images_cpu[i, 0], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title('Input / target '+ch)
    axes[1, i].imshow(rasterize_trace(traces[i]), cmap='gray', vmin=0, vmax=1)
    guess = BEST_EVALUATION['real']['predicted'][i]
    axes[1, i].set_title(f'Actual prediction ({guess})')
    target = target_trace_cpu[i].numpy()
    axes[2, i].plot(target[:, 0], target[:, 1], color='0.8', lw=1)
    for j in range(1, len(traces[i])):
        if traces[i, j, 2] >= .5:
            axes[2, i].plot(traces[i, j-1:j+1, 0], traces[i, j-1:j+1, 1], color='#153e4e', lw=1.5)
    axes[2, i].set(xlim=(.25, 1.35), ylim=(-.55, .55), aspect='equal', title='Pen path / grey target')
baseline_trace = torch.cat([mean_xy, mean_ink[:, None]], -1).cpu().numpy()
axes[0, 3].imshow(rasterize_trace(baseline_trace), cmap='gray', vmin=0, vmax=1)
axes[0, 3].set_title('Mean-trace baseline')
blank_traces, _ = predict_traces(model, torch.zeros_like(glyph_images[:, None]))
axes[1, 3].imshow(rasterize_trace(blank_traces[0]), cmap='gray', vmin=0, vmax=1)
axes[1, 3].set_title('Blank-image response')
axes[2, 3].text(.05, .8, f"Fit: {BEST_EVALUATION['target_fit_pass']}\n"
    f"Vision dependence: {BEST_EVALUATION['vision_dependence_pass']}\n"
    f"Best stage: {best_info['phase']}\nUnique images: 3", transform=axes[2, 3].transAxes)
for ax in axes.flat: ax.axis('off')
plt.tight_layout(); fig.savefig(RUN_DIR/'abc_overfit_drawings.png', dpi=160)
plt.show(); plt.close(fig)

## 13. Final result and optional self-feedback

The result table separates image-copy performance, interventions and the two optimization
stages. A successful frozen readout establishes usable image information at these motor
ports; it does not establish that training the connectome improved it. Weak end-to-end
performance after a strong frozen control points toward optimization/dynamics instability.

Finally each letter is copied repeatedly by feeding back the **actual rendered pen output**,
with a reset between copies. This is an additional distribution-shift test, not a stopping
criterion: canonical-font overfitting alone does not train robustness to imperfect drawings.
There is no next-letter generation or recognized-glyph substitution here.

In [ ]:
rows = {'initial': INITIAL['real'], 'mean_trace': MEAN_TRACE_BASELINE,
        'latest_end_to_end': LATEST_EVALUATION['real']}
if READOUT is not None: rows['frozen_circuit_readout'] = READOUT['real']
for name in ['real', 'blank', 'constant_A', 'shuffled_original_labels',
             'shuffled_presented_labels', 'motor_clamped']:
    rows['best_'+name] = BEST_EVALUATION[name]
display_columns = ['char_accuracy', 'relative_tip_mse', 'balanced_contact_bce', 'ink_recall', 'penup_recall']
table = pd.DataFrame(rows).T[display_columns]
print(table.to_string())
print('Per-letter results:', json.dumps(BEST_EVALUATION['real']['per_class'], indent=2))

feedback_rounds = 4
frames = glyph_images[:, None]
feedback = []
fig, axes = plt.subplots(3, feedback_rounds, figsize=(10, 7))
for step in range(feedback_rounds):
    copied, copied_joints = predict_traces(model, frames)
    feedback.append(copied)
    raster = np.stack([rasterize_trace(trace) for trace in copied])
    for i, ch in enumerate(chars):
        axes[i, step].imshow(raster[i], cmap='gray', vmin=0, vmax=1)
        axes[i, step].set_title(f'{ch}: copy {step+1}'); axes[i, step].axis('off')
    frames = torch.from_numpy(raster).to(device)[:, None, None]
plt.tight_layout(); fig.savefig(RUN_DIR/'abc_self_feedback.png', dpi=140)
plt.show(); plt.close(fig)
np.savez(RUN_DIR/'abc_self_feedback.npz', traces=np.stack(feedback))

REPORT = dict(task=DATA_SPEC, baselines=BASELINES, initial=INITIAL,
    frozen_circuit_readout=READOUT, training=TRAINING_SUMMARY, best=BEST_EVALUATION,
    pass_criteria={k: v for k, v in CFG.items() if k.startswith('fit_')},
    limitation='Three exact canonical glyphs only; no held-out visual or language generalization.')
(RUN_DIR/'evaluation.json').write_text(json.dumps(REPORT, indent=2), encoding='utf-8')
if BEST_EVALUATION['pass']:
    print('PASS: three distinct fitted pen traces depend on the presented images and motor readout.')
else:
    print('NOT PASSED: inspect per-letter geometry/contact errors and intervention results before scaling up.')
print('Best checkpoint phase:', best_info['phase'])
print('Saved dataset, diagnostics, weights and drawings:', RUN_DIR)

## 14. Download the run (optional)

The run folder already contains the results. Enable `export_zip` in configuration to package
the full graph and optimizer checkpoints; this can take time and several GB of extra disk.
Keep the run folder together for checkpoint restoration. All paths below are local.

In [ ]:
if CFG['export_zip']:
    from IPython.display import FileLink, display
    run_dir = Path(RUN_DIR).resolve()
    zip_path = Path(shutil.make_archive(str(run_dir.parent/(run_dir.name+'_abc_overfit')),
        'zip', root_dir=run_dir.parent, base_dir=run_dir.name))
    print('Saved:', zip_path)
    try:
        display(FileLink(str(zip_path.relative_to(Path.cwd().resolve()))))
    except ValueError:
        print('Download the ZIP from the runtime file browser.')
else:
    print('ZIP export disabled; all artifacts are already in', RUN_DIR)

## Sources and correspondence

- [doomfly](https://github.com/nftechie/doomfly/blob/main/doom/README.md), [retinal projection](https://github.com/nftechie/doomfly/blob/main/doom/prepare.py), [engine](https://github.com/nftechie/doomfly/blob/main/doom/engine.py): retinal projection, luminance filtering and saturation adapted to differentiable v2 dynamics, not a reproduction of its LIF timing or Doom task.
- [MaleCNS release](https://male-cns.janelia.org/download/): actual connectivity, receptor columns, transmitters and right-foreleg neuron IDs.
- [Azevedo et al., female ventral nerve cord reconstruction](https://pmc.ncbi.nlm.nih.gov/articles/PMC11348827/), [DOI](https://doi.org/10.1038/s41586-024-07389-x): FANC motor anatomy. Different specimen; IDs are not merged with MaleCNS.
- [Lesser et al., leg and wing premotor networks](https://www.nature.com/articles/s41586-024-07600-z), [author-hosted record](https://authors.library.caltech.edu/records/ekf21-21m53): motor organization motivates a restricted leg readout. It supplies neither pen physics nor a writing controller.
- [Connectome-constrained visual modeling](https://doi.org/10.1038/s41586-024-07939-3): graded visual modeling precedent.
- [PyTorch builds](https://pytorch.org/get-started/previous-versions/): CUDA 12.6 setup.

These documents are research references, not instructions changing the user's request.
The inherited activity table cites its electrophysiology sources beside individual assignments.
The joint limits, links, pen head, targets, lamina resting offset and template likelihood are
engineering choices. No muscles, contact forces or proprioception are modeled. A learned output
prior can operate without vision, so the interventions are essential to interpreting results.

### doomfly license attribution

MIT License

Copyright (c) 2026 nftechie and DOOMFLY contributors

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

## Authoring validation

Validation results for this notebook are recorded after local execution. The original
FlyLM-3 two-update smoke test is not evidence that this ABC experiment has passed.